### Read Metadata

In [0]:
from pyspark.sql.functions import *

metadata_df = spark.table(
    "cicd_dev.silver.silver_metadata"
)

display(metadata_df)

### Create Generic Transformation Function

In [0]:
from pyspark.sql.functions import *

def apply_dq_rules(df):

    # Remove duplicates
    df = df.dropDuplicates()

    # Remove rows where all columns are null
    df = df.na.drop(how="all")

    return df

### Table-Specific Rules

In [0]:
def apply_business_rules(df, table_name):

    if table_name == "customers":

        if "customer_id" in df.columns:
            df = df.filter(
                col("customer_id").isNotNull()
            )

        if "city" in df.columns:
            df = df.withColumn(
             "city",
             trim(
            upper(col("city"))
            )
         )

    elif table_name == "sales":

        if "sales_amount" in df.columns:
            df = df.filter(
                col("sales_amount") > 0
            )

    elif table_name == "products":

        if "price" in df.columns:
            df = df.filter(
                col("price") > 0
            )

    return df

In [0]:
metadata = metadata_df.collect()

for row in metadata:

    bronze_table = row["bronze_table"]
    silver_table = row["silver_table"]

    source_table = f"cicd_dev.bronze.{bronze_table}"
    target_table = f"cicd_dev.silver.{silver_table}"

    try:

        print(f"Processing {bronze_table}")

        df = spark.table(source_table)

        transformed_df = apply_dq_rules(df)

        transformed_df = apply_business_rules(
            transformed_df,
            silver_table
        )

        transformed_df.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(target_table)

        print(f"Loaded {silver_table}")

    except Exception as e:

        print(
            f"Failed {silver_table} : {str(e)}"
        )

In [0]:
%sql
SHOW TABLES IN cicd_dev.silver;

In [0]:
%sql
SELECT *
FROM cicd_dev.silver.customers;